In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import plotly.express as px
import warnings 
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("diabetes_prediction_dataset.csv")
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
df.shape

(100000, 9)

In [4]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'],
      dtype='object')

In [5]:
df.dtypes

gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object

In [6]:
df.isnull().sum()

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

In [7]:
df["hypertension"].value_counts()

hypertension
0    92515
1     7485
Name: count, dtype: int64

In [8]:
df["heart_disease"].value_counts()

heart_disease
0    96058
1     3942
Name: count, dtype: int64

In [9]:
df["HbA1c_level"].value_counts()

HbA1c_level
6.6    8540
5.7    8413
6.5    8362
5.8    8321
6.0    8295
6.2    8269
6.1    8048
3.5    7662
4.8    7597
4.5    7585
4.0    7542
5.0    7471
8.2     661
8.8     661
9.0     654
7.5     643
6.8     642
7.0     634
Name: count, dtype: int64

In [10]:
df["smoking_history"].value_counts()

smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64

In [11]:
num_cols=df.select_dtypes(include=["int64","float64"]).columns
cat_cols=df.select_dtypes(include=["object"]).columns
print("Categorical Columns")
print(cat_cols)
print("------------------==============")
print("Numerical Columns")
print(num_cols)

Categorical Columns
Index(['gender', 'smoking_history'], dtype='object')
------------------==============
Numerical Columns
Index(['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level',
       'blood_glucose_level', 'diabetes'],
      dtype='object')


In [12]:
## Target variable
df["diabetes"].value_counts()

diabetes
0    91500
1     8500
Name: count, dtype: int64

#### From that i understand this is a imbalanced dataset why because look 0 is very much higher like 90-10 ..So this is a Imbalanced dataset.So i want to do either randomUnderSampling but we want to lose the dat and RandomOverSampling it increase the data and SmOte approach 

In [13]:
#Outliers
num_cols=["age","bmi","HbA1c_level","blood_glucose_level"]
for col in num_cols:
    fig=px.box(
        df,
        y=col,
        title=f"Boxplot of {col}",
        points="outliers")
    fig.show()

In [14]:
#Corelation
corr=df.corr(numeric_only=True)
fig=px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="Rdbu_r",
    title="Corelation Matrix")
fig.update_layout(
    width=1000,
    height=500)
fig.show()

In [15]:
## Task -2 Prepare the data
cat_cols

Index(['gender', 'smoking_history'], dtype='object')

In [16]:
df_clean=df.copy()

In [17]:
df_clean=df_clean[df_clean["gender"]!="Other"]

In [18]:
df_clean["smoking_history"]=df_clean["smoking_history"].replace("No Info","never")

In [19]:
df_clean=pd.get_dummies(
    df_clean,
    columns=["gender","smoking_history"],
    drop_first=True)


In [20]:
df_clean.head()

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes,gender_Male,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
0,80.0,0,1,25.19,6.6,140,0,False,False,False,True,False
1,54.0,0,0,27.32,6.6,80,0,False,False,False,True,False
2,28.0,0,0,27.32,5.7,158,0,True,False,False,True,False
3,36.0,0,0,23.45,5.0,155,0,False,False,False,False,False
4,76.0,1,1,20.14,4.8,155,0,True,False,False,False,False


In [21]:
X=df_clean.drop("diabetes",axis=1)
y=df_clean["diabetes"]

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [23]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
num_cols=["age","bmi","HbA1c_level","blood_glucose_level"]
X_train[num_cols]=scaler.fit_transform(X_train[num_cols])
X_test[num_cols]=scaler.transform(X_test[num_cols])

In [30]:
## Task-3 Feature Enginerring
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
importances=rf.feature_importances_
feature_importance=(
    pd.DataFrame({
        "feature":X_train.columns,
        "importance":importances})
    .sort_values(by="importance",ascending=False)
    )
print(feature_importance)

                        feature  importance
4                   HbA1c_level    0.410676
5           blood_glucose_level    0.314407
3                           bmi    0.125861
0                           age    0.101112
1                  hypertension    0.014095
2                 heart_disease    0.010666
6                   gender_Male    0.007755
9         smoking_history_never    0.004956
8        smoking_history_former    0.004464
10  smoking_history_not current    0.003186
7          smoking_history_ever    0.002822


In [31]:
top_features=feature_importance["feature"].head(5).tolist()
top_features

['HbA1c_level', 'blood_glucose_level', 'bmi', 'age', 'hypertension']

In [32]:
X_train_top=X_train[top_features]
X_test_top=X_test[top_features]


In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
model=LogisticRegression(class_weight="balanced",max_iter=1000)
model.fit(X_train,y_train)
y_pred_full=model.predict(X_test)
print("Full Features:")
print(classification_report(y_test,y_pred_full))
#Top Feature
model.fit(X_train_top,y_train)
y_pred_top=model.predict(X_test_top)
print("TOP Features:\n")
print(classification_report(y_test,y_pred_top))

Full Features:
              precision    recall  f1-score   support

           0       0.99      0.89      0.94     18297
           1       0.42      0.88      0.57      1700

    accuracy                           0.89     19997
   macro avg       0.71      0.89      0.75     19997
weighted avg       0.94      0.89      0.90     19997

TOP Features:

              precision    recall  f1-score   support

           0       0.99      0.89      0.93     18297
           1       0.42      0.89      0.57      1700

    accuracy                           0.89     19997
   macro avg       0.70      0.89      0.75     19997
weighted avg       0.94      0.89      0.90     19997



In [37]:
X_train_use=X_train_top
X_test_use=X_test_top

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,roc_auc_score,classification_report,f1_score

In [51]:
## Baseline Model
lr=LogisticRegression(class_weight="balanced",max_iter=1000)
lr.fit(X_train_use,y_train)
y_pred_lr=lr.predict(X_test_use)
y_prob_lr=lr.predict_proba(X_test_use)[:,1]
print("Logistic Regression")
print(classification_report(y_test,y_pred_lr))
print("ROC-AUC:",roc_auc_score(y_test,y_prob_lr))
print("Precision:",precision_score(y_test,y_pred_lr))
print("Recall_Score:",recall_score(y_test,y_pred_lr))

Logistic Regression
              precision    recall  f1-score   support

           0       0.99      0.89      0.93     18297
           1       0.42      0.89      0.57      1700

    accuracy                           0.89     19997
   macro avg       0.70      0.89      0.75     19997
weighted avg       0.94      0.89      0.90     19997

ROC-AUC: 0.9608406874801076
Precision: 0.41759156492785793
Recall_Score: 0.8852941176470588


In [50]:
dt=DecisionTreeClassifier(class_weight="balanced",random_state=42)
dt.fit(X_train_use,y_train)
y_pred_dt=dt.predict(X_test_use)
y_prob_dt=dt.predict_proba(X_test_use)[:,1]
print("Decision Tree")
print(classification_report(y_test,y_pred_dt))
print("ROC-AUC",roc_auc_score(y_test,y_prob_dt))
print("Precision:",precision_score(y_test,y_pred_dt))
print("Recall",recall_score(y_test,y_pred_dt))

Decision Tree
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     18297
           1       0.68      0.74      0.70      1700

    accuracy                           0.95     19997
   macro avg       0.83      0.85      0.84     19997
weighted avg       0.95      0.95      0.95     19997

ROC-AUC 0.8549532871026752
Precision: 0.6764069264069265
Recall 0.7352941176470589


In [48]:
rf=RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42)
rf.fit(X_train_use,y_train)
y_pred_rf=rf.predict(X_test_use)
y_prob_rf=rf.predict_proba(X_test_use)[:,1]
print("random Forest")
print(classification_report(y_test,y_pred_rf))
print("ROC_AUC",roc_auc_score(y_test,y_prob_rf))
print("Precision",precision_score(y_test,y_pred_rf))
print("Recall",recall_score(y_test,y_pred_rf))

random Forest
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18297
           1       0.85      0.70      0.77      1700

    accuracy                           0.96     19997
   macro avg       0.91      0.84      0.88     19997
weighted avg       0.96      0.96      0.96     19997

ROC_AUC 0.9613852158341611
Precision 0.8542713567839196
Recall 0.7


In [54]:
res=[]
def evaluate(name,y_test,y_pred,y_prob):
    res.append({
        "Model":name,
        "Precision":precision_score(y_test,y_pred),
         "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
evaluate("Logistic", y_test, y_pred_lr, y_prob_lr)
evaluate("Decision Tree", y_test, y_pred_dt, y_prob_dt)
evaluate("Random Forest", y_test, y_pred_rf, y_prob_rf)
print(pd.DataFrame(res))   

           Model  Precision    Recall        F1   ROC-AUC
0       Logistic   0.417592  0.885294  0.567496  0.960841
1  Decision Tree   0.676407  0.735294  0.704622  0.854953
2  Random Forest   0.854271  0.700000  0.769479  0.961385


In [56]:
## tas-5
## SMOte
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
X_train_sm,y_train_sm=smote.fit_resample(X_train_use,y_train)
print("Before",y_train.value_counts())
print("After:\n",y_train_sm.value_counts())

Before diabetes
0    73185
1     6800
Name: count, dtype: int64
After:
 diabetes
0    73185
1    73185
Name: count, dtype: int64


In [59]:
## Baseline model
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=100)
rf.fit(X_train_sm,y_train_sm)
y_pred_rf=rf.predict(X_test_use)
y_prob_rf=rf.predict_proba(X_test_use)[:,1]
print("Random Forest/n")
print(classification_report(y_test,y_pred_rf))
print("ROC_AUC",roc_auc_score(y_test,y_prob_rf))
print("Precision",precision_score(y_test,y_pred_rf))
print("Recall:",recall_score(y_test,y_pred_rf))

Random Forest/n
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     18297
           1       0.77      0.74      0.75      1700

    accuracy                           0.96     19997
   macro avg       0.87      0.86      0.87     19997
weighted avg       0.96      0.96      0.96     19997

ROC_AUC 0.963361656845063
Precision 0.7695144437615242
Recall: 0.7364705882352941


In [66]:
from sklearn.model_selection import GridSearchCV
param_grid={
    "n_estimators":[100,200,300],
    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]}
rf_base=RandomForestClassifier(random_state=42)
grid=GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    verbose=1,
    n_jobs=-1)
grid.fit(X_train_sm,y_train_sm)
best_rf=grid.best_estimator_
print("Best Parameter:",grid.best_params_)


Fitting 3 folds for each of 108 candidates, totalling 324 fits
Best Parameter: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [68]:
y_pred_best=best_rf.predict(X_test_use)
y_prob_best=best_rf.predict_proba(X_test_use)[:,1]
print("Tuned Random Forest")
print(classification_report(y_test,y_pred_best))
print("ROC_AUC",roc_auc_score(y_test,y_prob_best))

Tuned Random Forest
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     18297
           1       0.78      0.74      0.76      1700

    accuracy                           0.96     19997
   macro avg       0.88      0.86      0.87     19997
weighted avg       0.96      0.96      0.96     19997

ROC_AUC 0.9701934100415048


In [74]:
thresholds=np.arange(0.3,0.5,0.6)
res=[]
for t in thresholds:
    y_pred_t=(y_prob_best>t).astype(int)
    res.append({
    "threshold":t,
    "precision":precision_score(y_test,y_pred_t),
    "recall":recall_score(y_test,y_pred_t),
    "f1":f1_score(y_test,y_pred_t)
    })
res_df=pd.DataFrame(res)
print(res_df)

   threshold  precision    recall        f1
0        0.3   0.587832  0.807059  0.680218


In [75]:
best_threshold = 0.6  
y_pred_final = (y_prob_best > best_threshold).astype(int)
print("\nFinal Model with Tuned Threshold\n")
print(classification_report(y_test, y_pred_final))


Final Model with Tuned Threshold

              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18297
           1       0.86      0.71      0.78      1700

    accuracy                           0.97     19997
   macro avg       0.91      0.85      0.88     19997
weighted avg       0.96      0.97      0.96     19997



In [77]:
## task -06
importances=rf.feature_importances_
feature_importance=pd.DataFrame({
    "feature":X_train_use.columns,
    "importance":importances}).sort_values(by="importance",ascending=False)
print(feature_importance.head(10))

               feature  importance
0          HbA1c_level    0.395270
1  blood_glucose_level    0.304760
3                  age    0.189052
2                  bmi    0.100364
4         hypertension    0.010554


In [79]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred_final)
print(cm)

[[18094   203]
 [  490  1210]]
